In [57]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV,LassoCV,ElasticNetCV


# load dataset

df_train = pd.read_csv('../data/kaggle_house_prices_train.csv')
df_test = pd.read_csv('../data/kaggle_house_prices_test.csv')

# Save test IDs and target before combining
test_ids = df_test['Id'].copy()
df_train['SalePrice'] = df_train['SalePrice'].apply(lambda x: np.log1p(x))
# y = np.log1p(df_train['SalePrice'])

In [58]:

# Preprocess , feature engineering and one-hot encoding
def fill_missing(df:pd.DataFrame,cat_none_cols:list[str],num_none_cols:list[str]):
    # Fill  categorical NAs with "None"
    for col in cat_none_cols:
        if col in df.columns:
            df[col] = df[col].fillna("None")
     # Fill  Numerical NAs with 0
    for col in num_none_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Fill remaining categorical NAs with mode
    for col in df.select_dtypes(include='object').columns:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].mode()[0])

    # Fill remaining numerical NAs with mean
    for col in df.select_dtypes(include='number').columns:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

    return df

def engineer_feature(df:pd.DataFrame):
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBathrooms'] = df['FullBath'] + (df['HalfBath'] * 0.5 ) + df['BsmtFullBath'] + (df['BsmtHalfBath'] * 0.5)
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
    df['WasRemodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
    df['SizeQuality'] = df['TotalSF'] * df['OverallQual']

    skewness = df.select_dtypes(include='number').skew().sort_values(ascending=False)

    # Filter to highly skewed
    high_skew = skewness[skewness.abs() > 0.75].index.tolist()

    # Remove target column
    if 'SalePrice' in high_skew:
        high_skew.remove('SalePrice')

    df[high_skew] = df[high_skew].apply(lambda x: np.log1p(x))

    return df


# Combine train and test data before preprocessing

df_train_size = len(df_train)

combined_df = pd.concat([df_train,df_test],axis=0).reset_index(drop=True)

# categorical NAs columns where NAs means "None"
cat_none_cols = [
    'PoolQC','MiscFeature','Alley','Fence','MasVnrType','FireplaceQu','GarageType','GarageFinish','GarageQual','GarageCond','BsmtFinType2','BsmtExposure','BsmtFinType1','BsmtCond','BsmtQual',
]

# numerical NAs columns where NAs means 0
num_none_cols = [
'GarageYrBlt','MasVnrArea','BsmtFullBath','BsmtHalfBath','GarageCars','GarageArea','TotalBsmtSF','BsmtUnfSF','BsmtFinSF2','BsmtFinSF1'
]
print(combined_df.shape)
print(f"Null columns in combined:{combined_df.isnull().sum().sum()}")
combined_df = fill_missing(combined_df,cat_none_cols,num_none_cols)
print(f"Null columns in combined:{combined_df.isnull().sum().sum()}")
combined_df = engineer_feature(combined_df)
combined_df = pd.get_dummies(data=combined_df,drop_first=True)
print(combined_df.shape)


# split back
train_encoded = combined_df.iloc[:df_train_size].copy()
test_encoded = combined_df.iloc[df_train_size:].copy()

print(f"train encoded shape:{train_encoded.shape}")
print(f"test encoded shape:{test_encoded.shape}")


(2919, 81)
Null columns in combined:17166
Null columns in combined:0
(2919, 267)
train encoded shape:(1460, 267)
test encoded shape:(1459, 267)


/var/folders/vd/nny5h1c50w12z_q275b2qf4r0000gn/T/ipykernel_11927/2695864920.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


In [59]:
# fitting training data via pipeline

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

X = train_encoded.drop(columns=['SalePrice','Id'],errors='ignore')
y = train_encoded['SalePrice']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# Use Pipeline for scaling to avoid data leakage

base_pipeline = Pipeline([('scaler', StandardScaler())])

# Smaller alphas — less aggressive regularisation
alphas = np.logspace(-4, -1, 50)   # 0.0001 to 0.1

models = {
    'Ridge CV':      RidgeCV(alphas=[0.01,0.1,1,10,100,1000], cv=5),
    'Lasso CV':      LassoCV(cv=5, max_iter=50000, n_jobs=-1,alphas=alphas),
    'ElasticNet CV': ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], cv=5, max_iter=50000, n_jobs=-1,alphas=alphas),
}

# Training using XGBRegressor


xgbModel = XGBRegressor(learning_rate=0.05, max_depth=4,n_estimators=400,random_state=42,verbosity=0,subsample=0.8,colsample_bytree=0.7
)
xgbModel.fit(X_train,y_train)
y_xgb_preds = xgbModel.predict(X_test)

xgb_cv_score =  cross_val_score(xgbModel,X_train,y_train,cv=5,scoring='neg_root_mean_squared_error',n_jobs=-1)

rows = []
for name,model in models.items():
     # Step 1 — build pipeline for this model
    pipe = clone(base_pipeline) 
    pipe.steps.append(('model', model))
    
    # cross validation
    cv_score = cross_val_score(model, X_train, y_train,  cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)

    # Step 3 — fit on full X_train and evaluate on X_test
    pipe.fit(X_train, y_train)
    y_pred    = pipe.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_r2   = r2_score(y_test, y_pred)
    best_alphas = pipe.named_steps['model'].alpha_

     # See how many features survived Lasso
    if name == 'Lasso CV':
        n_features = (pipe.named_steps['model'].coef_ != 0).sum()
        print(f"Features kept = {n_features} | Total features = {len(X.columns)}")

    # See best l1_ratio of ElasticNet
    if name == 'ElasticNet CV':
        best_l1_ratio = pipe.named_steps['model'].l1_ratio_
        print(f"Best l1 ratio = {best_l1_ratio}")

    # collect results
    rows.append([name, (-cv_score).mean(), (-cv_score).std(), test_rmse, test_r2,best_alphas])

    print(f"{name:12} CV RMSE={(-cv_score).mean():.4f} | Test RMSE={test_rmse:.4f} | R²={test_r2:.4f}")

# add XGB result
rows.append(['XGBRegressor',(-xgb_cv_score).mean(), (-xgb_cv_score).std(),np.sqrt(mean_squared_error(y_test,y_xgb_preds)),r2_score(y_test, y_xgb_preds)])

# print comparison table
summary = pd.DataFrame(rows, columns=['Model', 'CV RMSE', 'CV Std', 'Test RMSE', 'R²','Best alphas'])
print("\n── Model comparison ─────────────────────────")
print(summary.round(4).to_string(index=False))


Ridge CV     CV RMSE=0.1332 | Test RMSE=0.1505 | R²=0.8786
Features kept = 66 | Total features = 265
Lasso CV     CV RMSE=0.1320 | Test RMSE=0.1421 | R²=0.8918
Best l1 ratio = 0.1
ElasticNet CV CV RMSE=0.1324 | Test RMSE=0.1430 | R²=0.8905

── Model comparison ─────────────────────────
        Model  CV RMSE  CV Std  Test RMSE     R²  Best alphas
     Ridge CV   0.1332  0.0197     0.1505 0.8786    1000.0000
     Lasso CV   0.1320  0.0209     0.1421 0.8918       0.0069
ElasticNet CV   0.1324  0.0202     0.1430 0.8905       0.0754
 XGBRegressor   0.1244  0.0124     0.1343 0.9033          NaN
